# SSLimPy — Getting Started and the Cosmological Calculator

SSLimPy (Single-dish Spectral Line intensity Mapping with Python) computes
line-intensity-mapping signals, their power spectra and covariances, and can be
used as a standalone cosmology, halo-model and line-emission calculator.

This first notebook shows the *global* SSLimPy setup and how to use the code as
a cosmology calculator. Run the notebook from the `notebooks/` directory.

```
from SSLimPy.interface import sslimpy
myssl = sslimpy.SSLimPy(settings_dict=settings, cosmopars=cosmodict, ...)
myssl.compute(cosmopars, halopars, astropars, obspars, ...)   # -> {"Power spectrum": ..., "Covariance": ...}
```

In [ ]:
import os
import sys

sys.path.append("../")

# Use the available cores for the Einstein--Boltzmann solver (CAMB/CLASS) and numba
os.environ.setdefault("OMP_NUM_THREADS", "12")

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

Cs = sns.color_palette("colorblind")

import niceplots
niceplots.utils.initPlot()

## 1. Global settings

`settings` are the *global* settings of a run: which Einstein--Boltzman solver to
use, whether to include RSD, the output one wants, etc. See the `Configuration`
class in `SSLimPy/interface/config.py` for the full list.

In [ ]:
settings = {
    "code": "camb",                # Einstein--Boltzmann solver: "camb" or "class"
    "do_RSD": True,                # include redshift-space distortions
    "nonlinearRSD": True,          # add fingers-of-god
    "QNLpowerspectrum": True,      # dewiggled (Zel'dovich) non-linear power spectrum
    "halo_model_PS": False,        # halo-model shot noise instead of Poissonian
    "nonlinearMatpow": True,       # store the non-linear matter power from the solver (halofit)
    "FoG_damp": "ISTF_like",       # parametrisation of the FoG damping, see power_spectrum.py
    "output": ["Power spectrum", "Covariance"],
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 10 * u.Mpc**-1,
    "nk": 50,
}

## 2. Cosmological parameters

Pass only the parameters you care about — the rest are filled with the fiducial
values (type hints and defaults live in `SSLimPy/cosmology/cosmology.py`).

In [ ]:
cosmodict = {
    "h": 0.677,
    "Omegam": 0.309177,
    "Omegab": 0.04903,
    "sigma8": 0.8222,
    "ns": 0.96824,
    "mnu": 0.06,
    "Neff": 3.044,
}

## 3. The SSLimPy object

The constructor builds the fiducial cosmology, halo model, astrophysics and
survey-specification objects. The first call is slow because CAMB/CLASS run and
the numba kernels compile.

In [ ]:
from SSLimPy.interface import sslimpy

myssl = sslimpy.SSLimPy(settings_dict=settings, cosmopars=cosmodict)

In [ ]:
mycosmo = myssl.current_cosmology   # the fiducial cosmology calculator
mycosmo.cosmopars

## 4. Distances and the Hubble rate

In [ ]:
z = np.linspace(0, 5, 200)

H = mycosmo.Hubble(z, physical=True).to(u.km / u.s / u.Mpc)
Dc = mycosmo.comoving(z)
DA = mycosmo.angdist(z)

fw, fh = plt.rcParams['figure.figsize']
fig, axs = plt.subplots(1, 2, figsize=(2 * fw, 1 * fh))
axs[0].plot(z, H, c=Cs[0])
axs[1].plot(z, Dc, c=Cs[1], label="comoving")
axs[1].plot(z, DA, c=Cs[2], label="angular diameter")

axs[0].set_xlabel(r"$z$")
axs[0].set_ylabel(r"$H(z)\,[\mathrm{km\,s^{-1}\,Mpc^{-1}}]$")
axs[1].set_xlabel(r"$z$")
axs[1].set_ylabel(r"$D\,[\mathrm{Mpc}]$")
axs[1].legend()
plt.tight_layout()

## 5. The matter power spectrum

`matpow(k, z, tracer=..., nonlinear=...)` returns the matter power spectrum on
your grid. `tracer="clustering"` refers to the CDM+baryons fluid; `"matter"`
includes neutrinos.

In [ ]:
k = np.geomspace(1e-3, 3, 200) * u.Mpc**-1
z0 = 0.0

Plin = mycosmo.matpow(k, z0, tracer="clustering", nonlinear=False)
Pnl = mycosmo.matpow(k, z0, tracer="clustering", nonlinear=True)

plt.loglog(k, Plin, c=Cs[0], label="linear")
plt.loglog(k, Pnl, c=Cs[1], ls="--", label="non-linear (halofit)")
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P_{\rm cb}(k)\,[\mathrm{Mpc}^{3}]$")
plt.legend()

## 6. Changing the cosmology on the fly

The `updater` module recreates the derived objects for the new parameters without
instantiating a fresh `SSLimPy`. This is exactly what `SSLimPy.compute(...)`
does internally when the passed dictionaries differ from the current ones.

In [ ]:
from copy import copy
from SSLimPy.interface import updater

cosmodict_hi = copy(mycosmo.cosmopars)
cosmodict_hi["Omegam"] = 0.34

cosmo_hi = updater.update_cosmo(mycosmo, cosmodict_hi, myssl.cfg)

In [ ]:
print("Omega_m =", mycosmo.Omega(0), " (fiducial)")
print("Omega_m =", cosmo_hi.Omega(0), " (modified)")

P_fid = mycosmo.matpow(k, z0, tracer="clustering", nonlinear=False)
P_hi = cosmo_hi.matpow(k, z0, tracer="clustering", nonlinear=False)

plt.loglog(k, P_fid, c=Cs[0], label=r"$\Omega_\mathrm{m}=0.309$")
plt.loglog(k, P_hi, c=Cs[1], ls="--", label=r"$\Omega_\mathrm{m}=0.34$")
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P_{\rm cb}(k)\,[\mathrm{Mpc}^{3}]$")
plt.legend()